# Green Hydrogen Lab Data Analysis Dashboard

Run this notebook in Google Colab. Upload `combined_wind_experiments.csv` when asked.

## Step 1 — Upload the dataset

In [ ]:
from google.colab import files
uploaded = files.upload()

## Step 2 — Prepare folders and load data

In [ ]:
import os, json, shutil
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

Path("data/raw").mkdir(parents=True, exist_ok=True)
Path("data/processed").mkdir(parents=True, exist_ok=True)
Path("outputs/figures").mkdir(parents=True, exist_ok=True)
Path("outputs/reports").mkdir(parents=True, exist_ok=True)

DATA_PATH = None
for filename in uploaded.keys():
    if filename.endswith(".csv"):
        target = f"data/raw/{filename}"
        shutil.move(filename, target)
        DATA_PATH = target
print("Dataset path:", DATA_PATH)

df_raw = pd.read_csv(DATA_PATH)
print("Rows, columns:", df_raw.shape)
display(df_raw.head())
print(df_raw.columns.tolist())

## Step 3 — Clean and standardize lab signals

In [ ]:
COLUMN_MAP = {
    "timestamp": "Unnamed: 0",
    "wind_power_kw": "Wind Turbine Power (kW)",
    "h2_flow": "IVAL_f_FM011_Flow",
    "specific_energy_kwh_per_kg": "Efficiency (kWh/kg)",
    "psu_voltage_vdc": "Power Supply Average Voltage (Vdc)",
    "psu_power_kw": "H2E_n_PSU_A_Power",
    "psu_current_a": "H2E_f_PSU_A_Current",
    "calc_h2_prod_rate": "H2E_f_Elec_CalcProdRate",
    "lfl_percent": "H2E_f_CG220_LFLPer",
    "pressure_pt307": "H2E_f_PT307_Pressure",
    "temp_te218_c": "H2E_f_TE218_Temp",
    "temp_te219_c": "H2E_f_TE219_Temp",
    "experiment": "Experiment",
}

keep = [v for v in COLUMN_MAP.values() if v in df_raw.columns]
df = df_raw[keep].copy().rename(columns={v:k for k,v in COLUMN_MAP.items()})
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
df["sample_index"] = np.arange(len(df))

for col in df.columns:
    if col not in ["timestamp", "experiment"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

df["experiment"] = df["experiment"].fillna("unknown_experiment")

for col in ["wind_power_kw","h2_flow","specific_energy_kwh_per_kg","psu_voltage_vdc","psu_power_kw","psu_current_a","calc_h2_prod_rate"]:
    if col in df.columns:
        df.loc[df[col] < 0, col] = np.nan

display(df.head())
df.info()

## Step 4 — Calculate green hydrogen KPIs

In [ ]:
# Assumption: rows are sequential samples, approximately 1 second each.
df["energy_kwh_est"] = df["psu_power_kw"] / 3600.0
df["h2_mass_kg_est"] = df["energy_kwh_est"] / df["specific_energy_kwh_per_kg"]
df["power_tracking_ratio"] = df["psu_power_kw"] / df["wind_power_kw"].replace(0, np.nan)

kpis = {
    "Rows analyzed": len(df),
    "Number of experiments": df["experiment"].nunique(),
    "Average wind power (kW)": round(df["wind_power_kw"].mean(), 2),
    "Average electrolyzer PSU power (kW)": round(df["psu_power_kw"].mean(), 2),
    "Estimated total energy (kWh)": round(df["energy_kwh_est"].sum(), 2),
    "Estimated total H2 mass (kg)": round(df["h2_mass_kg_est"].sum(), 4),
    "Average specific energy (kWh/kg)": round(df["specific_energy_kwh_per_kg"].mean(), 2),
    "Median specific energy (kWh/kg)": round(df["specific_energy_kwh_per_kg"].median(), 2),
    "Average selected temperature (°C)": round(df["temp_te218_c"].mean(), 2),
    "Median power tracking ratio": round(df["power_tracking_ratio"].median(), 3),
}
kpi_df = pd.DataFrame(kpis.items(), columns=["KPI","Value"])
display(kpi_df)

## Step 5 — Plot power response

In [ ]:
plot_df = df.iloc[:min(len(df), 5000)]
plt.figure(figsize=(12,5))
plt.plot(plot_df["sample_index"], plot_df["wind_power_kw"], label="Wind turbine power (kW)", linewidth=1)
plt.plot(plot_df["sample_index"], plot_df["psu_power_kw"], label="Electrolyzer PSU power (kW)", linewidth=1)
plt.title("Wind Input vs Electrolyzer Power Response")
plt.xlabel("Sample index")
plt.ylabel("Power (kW)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("outputs/figures/01_power_response.png", dpi=180)
plt.show()

## Step 6 — Plot specific energy distribution

In [ ]:
s = df["specific_energy_kwh_per_kg"].dropna()
s = s[(s > 0) & (s < s.quantile(0.99))]
plt.figure(figsize=(9,5))
plt.hist(s, bins=40)
plt.title("Specific Energy Distribution")
plt.xlabel("kWh per kg H2")
plt.ylabel("Frequency")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("outputs/figures/02_specific_energy_distribution.png", dpi=180)
plt.show()

## Step 7 — Plot cumulative hydrogen production estimate

In [ ]:
plot_df = df.iloc[:min(len(df), 5000)].copy()
plot_df["h2_cumulative_kg_est"] = plot_df["h2_mass_kg_est"].cumsum()
plt.figure(figsize=(12,5))
plt.plot(plot_df["sample_index"], plot_df["h2_cumulative_kg_est"], linewidth=1.5)
plt.title("Estimated Cumulative Hydrogen Production")
plt.xlabel("Sample index")
plt.ylabel("Estimated H2 mass (kg)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("outputs/figures/03_cumulative_h2_production.png", dpi=180)
plt.show()

## Step 8 — Plot selected temperature signals

In [ ]:
plot_df = df.iloc[:min(len(df), 5000)]
plt.figure(figsize=(12,5))
plt.plot(plot_df["sample_index"], plot_df["temp_te218_c"], label="TE218 temperature", linewidth=1)
plt.plot(plot_df["sample_index"], plot_df["temp_te219_c"], label="TE219 temperature", linewidth=1)
plt.title("Selected Lab Temperature Signals")
plt.xlabel("Sample index")
plt.ylabel("Temperature (°C)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("outputs/figures/04_temperature_trends.png", dpi=180)
plt.show()

## Step 9 — Compare experiments

In [ ]:
exp = df.groupby("experiment").agg(
    samples=("experiment","size"),
    avg_power_kw=("psu_power_kw","mean"),
    avg_kwh_per_kg=("specific_energy_kwh_per_kg","mean"),
    total_h2_kg=("h2_mass_kg_est","sum")
).reset_index().sort_values("samples", ascending=False).head(12)
display(exp)

labels = [str(x).replace(".csv","") for x in exp["experiment"]]
plt.figure(figsize=(12,6))
plt.barh(labels, exp["avg_kwh_per_kg"])
plt.title("Average Specific Energy by Experiment")
plt.xlabel("Average kWh per kg H2")
plt.ylabel("Experiment")
plt.grid(True, axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig("outputs/figures/05_experiment_efficiency_comparison.png", dpi=180)
plt.show()

## Step 10 — Save outputs and download

In [ ]:
df.to_csv("data/processed/cleaned_green_hydrogen_lab_data.csv", index=False)
exp.to_csv("outputs/reports/experiment_summary.csv", index=False)
with open("outputs/reports/kpis.json", "w") as f:
    json.dump(kpis, f, indent=2)

report_lines = [
    "# Green Hydrogen Lab Data Analysis Dashboard",
    "",
    "## Dataset",
    "Public Reference Data for Megawatt-Scale Hydrogen Electrolysis, NLR Submission 305.",
    "",
    "## KPIs",
]
for k, v in kpis.items():
    report_lines.append(f"- {k}: {v}")
report_lines += [
    "",
]
with open("outputs/reports/analysis_report.md", "w") as f:
    f.write("\n".join(report_lines))

shutil.make_archive("green_hydrogen_lab_dashboard_outputs", "zip", ".")
files.download("green_hydrogen_lab_dashboard_outputs.zip")
print("Done. Download the ZIP and upload the project files to GitHub.")

> Result:
>
> Developed a Python-based green hydrogen lab data analysis dashboard using public megawatt-scale electrolysis data; cleaned and validated CSV time-series data, calculated hydrogen production and energy-efficiency KPIs, visualized operational trends with matplotlib, and documented the analysis pipeline for reproducible R&D reporting.